# 🌍 全球供应链风险与物流绩效分析
## Global Supply Chain Risk & Logistics Performance Analysis

**数据集**: Global Supply Chain Risk & Logistics (2024-2026) — Kaggle  
**数据量**: 5,000 条国际货运记录  
**分析工具**: pandas · numpy · matplotlib · seaborn · scikit-learn · scipy

---

### 分析框架

| 层次 | 维度 | 关键问题 |
|------|------|----------|
| 📦 描述分析 | 中断率 / 时效 / 风险因素 / 路线 | 哪些因素与中断相关？ |
| 📐 统计推断 | t 检验 / ANOVA / 卡方检验 | 组间差异是否显著？ |
| 🤖 预测建模 | Baseline / 逻辑回归 / 随机森林 | 能否预测？哪个模型最优？ |
| 🎲 蒙特卡洛 | Bootstrap / 压力测试 / VaR / 成本 | 风险有多大？置信区间？ |

## 1. 导入库与数据加载

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
from pathlib import Path

# 导入项目模块
sys.path.insert(0, str(Path.cwd() / 'src'))
from data_loader import prepare_data
from analysis import *
from statistical_tests import *
from monte_carlo import MonteCarloEngine, run_monte_carlo_analysis
from visualization import *

# 设置
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)
%matplotlib inline

In [ ]:
# 加载并预处理数据
df = prepare_data()

# 快速预览
print(f"\n数据维度: {df.shape}")
print(f"时间范围: {df['Date'].min().date()} ~ {df['Date'].max().date()}")
df.head(5)

## 2. 数据概览

了解数据的整体分布情况：数值特征统计、类别特征分布。

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Data Overview: Distribution of Key Features', fontsize=16, fontweight='bold')

# 运输模式
df['Transport_Mode'].value_counts().plot(kind='pie', autopct='%1.1f%%', ax=axes[0, 0],
    colors=CAT_PALETTE[:5], wedgeprops={'edgecolor': 'white', 'linewidth': 1.5})
axes[0, 0].set_ylabel('')
axes[0, 0].set_title('Transport Mode Distribution')

# 产品类别
df['Product_Category'].value_counts().plot(kind='barh', ax=axes[0, 1], color=CAT_PALETTE, edgecolor='white')
axes[0, 1].set_title('Product Category Distribution')

# 距离分布
df['Distance_km'].hist(bins=50, ax=axes[0, 2], color='#2E86AB', edgecolor='white', alpha=0.8)
axes[0, 2].axvline(df['Distance_km'].median(), color='red', linestyle='--', label=f"Median: {df['Distance_km'].median():.0f} km")
axes[0, 2].set_title('Distance Distribution')
axes[0, 2].legend()

# 时效分布
df['Lead_Time_Days'].hist(bins=50, ax=axes[1, 0], color='#A23B72', edgecolor='white', alpha=0.8)
axes[1, 0].axvline(df['Lead_Time_Days'].median(), color='blue', linestyle='--', label=f"Median: {df['Lead_Time_Days'].median():.1f}d")
axes[1, 0].set_title('Lead Time Distribution')
axes[1, 0].legend()

# 地缘政治风险
df['Geopolitical_Risk_Score'].hist(bins=20, ax=axes[1, 1], color='#F18F01', edgecolor='white', alpha=0.8)
axes[1, 1].set_title('Geopolitical Risk Score Distribution')

# 中断 vs 正常
df['Disruption_Occurred'].value_counts().plot(kind='pie', autopct='%1.1f%%', ax=axes[1, 2],
    labels=['Normal', 'Disrupted'], colors=['#3A7D44', '#D62828'],
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}, explode=[0, 0.05])
axes[1, 2].set_ylabel('')
axes[1, 2].set_title('Disruption vs Normal')

plt.tight_layout()
plt.show()

## 3. 📦 供应链中断分析

**核心问题**: 哪些运输方式、产品类别最容易受到中断的影响？中断是否有季节性规律？

In [ ]:
# 运行中断分析
disruption_results = disruption_summary(df)

# 可视化
plot_disruption_by_mode_and_product(df)

In [ ]:
# 天气条件对中断的影响
weather_disruption = disruption_by_weather(df)
print("天气条件详细影响:")
print(weather_disruption.to_string())
print(f"\n💡 关键发现: '{weather_disruption['Disruption_Rate'].idxmax()}' 天气下中断率最高 ({weather_disruption['Disruption_Rate'].max():.1f}%), "
      f"'{weather_disruption['Disruption_Rate'].idxmin()}' 天气下中断率最低 ({weather_disruption['Disruption_Rate'].min():.1f}%)")

## 4. ⏱️ 时效绩效分析

**核心问题**: 不同运输方式的时效差异有多大？哪些因素影响在途天数？

In [ ]:
# 运行时效分析
lt_results = lead_time_analysis(df)

# 可视化：箱线图
plot_lead_time_distribution(df)

## 5. 🔍 风险因素深度分析

**核心问题**: 地缘政治风险、天气条件、承运商可靠性——哪个对中断的影响最大？

In [ ]:
# 运行风险分析
risk_results = risk_factor_analysis(df)

# 可视化
plot_risk_vs_disruption(df)
plot_weather_impact(df)

## 6. 📊 相关性分析

**核心问题**: 各物流绩效指标之间的相关性如何？哪些因素与中断密切相关？

In [ ]:
# 相关性热力图
corr_matrix = correlation_analysis(df)
plot_correlation_heatmap(df)

## 7. 🌍 路线绩效分析

**核心问题**: 哪些国际路线的中断风险最高？时效最差？如何选择最优路线？

In [ ]:
# 路线分析
top_routes = route_analysis(df, top_n=15)

# 可视化：气泡图
plot_route_performance(df, top_n=15)

## 8. 🤖 中断预测模型

**核心问题**: 能否基于货运特征预测中断风险？哪些特征最重要？

In [ ]:
# 构建多模型对比（Baseline / 逻辑回归 / 随机森林）
model_results = build_disruption_model(df)

# 模型对比表
print("\n📋 模型对比汇总:")
display(model_results['model_comparison'])

# 随机森林特征重要性
print("\n🌲 Random Forest 特征重要性:")
display(model_results['feature_importance'])

# 逻辑回归系数（可解释性）
print("\n📐 逻辑回归系数:")
display(model_results['lr_feature_importance'])

# 可视化
plot_feature_importance(model_results['feature_importance'])
plot_model_comparison(model_results['model_comparison'])

## 9. 📐 统计假设检验

**核心问题**: 前面观察到的组间差异是否统计显著？效应量有多大？

- **t 检验**: 中断批次 vs 正常批次的时效差异
- **ANOVA**: 四种运输模式间时效差异
- **卡方检验**: 中断率与类别变量的独立性

In [ ]:
# 运行所有统计检验
test_results = run_statistical_tests(df)

# 效应量可视化
plot_statistical_results(test_results)

## 10. 🎲 蒙特卡洛模拟

**核心问题**: 中断率的真实范围是什么？在极端情景下风险会多严重？各路线的最坏时效是多少？

模拟情景：
- Bootstrap 估计中断率置信区间
- 地缘政治风险 +2.0 压力测试
- 恶劣天气 ×3 压力测试
- 路线级 Value-at-Risk（VaR 95% / 99%）
- 总供应链风险成本分布

In [ ]:
# 初始化蒙特卡洛引擎
mc_engine = MonteCarloEngine(df, random_seed=42)

# 运行完整蒙特卡洛分析（10,000 次模拟）
mc_results = run_monte_carlo_analysis(mc_engine, n_samples=10000)

# 可视化
plot_mc_disruption_distribution(mc_results)
plot_mc_stress_test(mc_results)
plot_route_var(mc_results)
plot_cost_distribution(mc_results)

## 12. 结论与建议

### 🔬 关键发现

**描述性分析**
1. **中断率**: 全样本约 61.3% 的货运经历中断，四种运输模式间差异<3pp
2. **时效**: 空运 1.6d / 公路 16.5d / 铁路 20.0d / 海运 39.8d — 差异巨大
3. **天气**: 飓风条件下中断率 100%，暴风雨 79.5%，晴天仅 37.0%

**统计推断**
1. 中断批次平均时效显著高于正常批次（t-test, p < 0.001）
2. 运输模式间时效差异高度显著（ANOVA, p < 0.001, η² 大效应量）
3. 中断率与天气条件强相关（卡方检验, p < 0.001, Cramér's V 大效应量）
4. 中断率与运输模式无显著关联（卡方检验不显著）— 说明"选择运输模式本身不能规避中断"

**预测建模**
1. 随机森林（Accuracy 66.0%, ROC-AUC 0.716）优于逻辑回归（63.1%, 0.635）和朴素基准（61.3%）
2. Top 3 特征：地缘政治风险 > 承运商可靠性 > Lead Time

**蒙特卡洛模拟**
1. 中断率 95% 置信区间通过 Bootstrap 量化
2. 地缘政治风险 +2.0 压力测试下中断率显著上升
3. 路线级 VaR 95% 揭示了高风险路线的最坏时效情景

### 💡 业务建议

- 🛡️ **地缘风险对冲**: 对高风险地区设置备选路线和应急库存
- 🚢 **承运商管理**: 可靠性评分是第二大可控因素，择优签约可降低 5-8% 中断
- 🌤️ **天气预警系统**: 飓风/暴风雨条件下启动应急预案
- 📊 **预测集成**: 将随机森林模型嵌入 TMS（运输管理系统）实现中断提前预警
- 🔄 **库存差异化**: 对纺织品等高中断率品类设置更高安全库存
- 📍 **路线优化**: 参考 VaR 报告，对高风险路线提前规划备选方案

## 11. 📋 综合仪表盘

一页汇总所有关键指标和分析结果。